# 04. End-to-End Evaluation

Ноутбук для финальной оценки всего пайплайна:
- viewpoint validation
- quality gate
- damage segmentation
- rule-based comparison `before / after`

Он не обучает модели, а использует готовые веса из проекта.

In [1]:

!pip install -q torch torchvision timm ultralytics pandas scikit-learn

In [2]:

import os
import sys
import json
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torchvision import transforms
import timm
from ultralytics import YOLO

In [3]:

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if any((cand / m).exists() for m in ["pyproject.toml", "requirements.txt", ".git"]):
            return cand
    return start.parent if start.parent.exists() else start

PROJECT_ROOT = find_project_root(Path("."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUALITY_VIEW_ROOT = PROJECT_ROOT / "ml" / "quality_view"
DAMAGE_SEG_ROOT = PROJECT_ROOT / "ml" / "damage_seg"
DATA_ROOT = PROJECT_ROOT / "ml" / "data"

VIEW_WEIGHTS = QUALITY_VIEW_ROOT / "weights" / "view_validation_best.pt"
QUALITY_WEIGHTS = QUALITY_VIEW_ROOT / "weights" / "quality_gate_best.pt"
SEG_WEIGHTS = DAMAGE_SEG_ROOT / "weights" / "best_damage_seg.pt"

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("DEVICE =", DEVICE)
print("VIEW_WEIGHTS   =", VIEW_WEIGHTS)
print("QUALITY_WEIGHTS=", QUALITY_WEIGHTS)
print("SEG_WEIGHTS    =", SEG_WEIGHTS)

DEVICE = mps
VIEW_WEIGHTS   = /Users/versache-karinndzi/Downloads/car-inspection-assistant/ml/quality_view/weights/view_validation_best.pt
QUALITY_WEIGHTS= /Users/versache-karinndzi/Downloads/car-inspection-assistant/ml/quality_view/weights/quality_gate_best.pt
SEG_WEIGHTS    = /Users/versache-karinndzi/Downloads/car-inspection-assistant/ml/damage_seg/weights/best_damage_seg.pt


In [5]:

def load_timm_checkpoint(ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model = timm.create_model(ckpt["backbone"], pretrained=False, num_classes=len(ckpt["classes"]))
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, ckpt

view_model, view_ckpt = load_timm_checkpoint(VIEW_WEIGHTS)
quality_model, quality_ckpt = load_timm_checkpoint(QUALITY_WEIGHTS)
seg_model = YOLO(str(SEG_WEIGHTS))

view_model.to(DEVICE)
quality_model.to(DEVICE)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/versache-karinndzi/Downloads/car-inspection-assistant/ml/damage_seg/weights/best_damage_seg.pt'

In [ ]:

view_tfms = transforms.Compose([
    transforms.Resize((view_ckpt["image_size"], view_ckpt["image_size"])),
    transforms.ToTensor(),
])

quality_tfms = transforms.Compose([
    transforms.Resize((quality_ckpt["image_size"], quality_ckpt["image_size"])),
    transforms.ToTensor(),
])

def predict_view(img_path: Path):
    img = Image.open(img_path).convert("RGB")
    x = view_tfms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = view_model(x).softmax(dim=1).cpu().numpy()[0]
    idx = int(np.argmax(probs))
    return {
        "pred_class": view_ckpt["classes"][idx],
        "confidence": float(probs[idx]),
        "all_probs": {cls: float(p) for cls, p in zip(view_ckpt["classes"], probs)},
        "accepted_as_valid": bool(probs[idx] >= view_ckpt["threshold"] and view_ckpt["classes"][idx] in {"front_valid", "rear_valid", "side_valid"}),
    }

def predict_quality(img_path: Path):
    img = Image.open(img_path).convert("RGB")
    x = quality_tfms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = quality_model(x).softmax(dim=1).cpu().numpy()[0]
    idx = int(np.argmax(probs))
    reject_idx = quality_ckpt["classes"].index("reject")
    reject_prob = float(probs[reject_idx])
    return {
        "pred_class": quality_ckpt["classes"][idx],
        "reject_prob": reject_prob,
        "accepted": bool(reject_prob < quality_ckpt["reject_threshold"]),
        "all_probs": {cls: float(p) for cls, p in zip(quality_ckpt["classes"], probs)},
    }

def predict_damage(img_path: Path, conf=0.25):
    preds = seg_model.predict(str(img_path), device=DEVICE, conf=conf, verbose=False)
    result = preds[0]
    detections = []
    names = result.names
    if result.boxes is None:
        return detections
    boxes = result.boxes
    for i in range(len(boxes)):
        cls_idx = int(boxes.cls[i].item())
        conf_i = float(boxes.conf[i].item())
        xyxy = [float(v) for v in boxes.xyxy[i].tolist()]
        detections.append({
            "class_name": names[cls_idx],
            "confidence": conf_i,
            "bbox_xyxy": xyxy,
        })
    return detections

## Формат сценарного eval-набора

Ожидаемая структура:

```text
ml/data/e2e_cases/
  case_001/
    before/
      front.jpg
      rear.jpg
      side.jpg
      extra_01.jpg
    after/
      front.jpg
      rear.jpg
      side.jpg
      extra_01.jpg
    meta.json
```

Где `meta.json` может содержать ground truth:
- `new_damage_expected: true/false`
- `notes`

In [ ]:

E2E_DIR = DATA_ROOT / "e2e_cases"
print("E2E_DIR =", E2E_DIR)

def summarize_image(img_path: Path, expected_slot: str | None = None):
    out = {
        "image_path": str(img_path),
        "view": None,
        "quality": None,
        "damage": None,
    }
    if expected_slot is not None:
        out["view"] = predict_view(img_path)
        out["slot_ok"] = out["view"]["accepted_as_valid"] and out["view"]["pred_class"].startswith(expected_slot)
    out["quality"] = predict_quality(img_path)
    out["damage"] = predict_damage(img_path)
    return out

def compare_detections(before_list, after_list, iou_distance_threshold=80.0):
    # Упрощённое rule-based сравнение по классу и расстоянию центров bbox.
    def center_xy(box):
        x1, y1, x2, y2 = box
        return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

    unmatched_after = []
    used_before = set()

    for a in after_list:
        ax, ay = center_xy(a["bbox_xyxy"])
        found = False
        for bi, b in enumerate(before_list):
            if bi in used_before:
                continue
            if a["class_name"] != b["class_name"]:
                continue
            bx, by = center_xy(b["bbox_xyxy"])
            dist = ((ax - bx)**2 + (ay - by)**2) ** 0.5
            if dist <= iou_distance_threshold:
                used_before.add(bi)
                found = True
                break
        if not found:
            unmatched_after.append(a)
    return unmatched_after

In [ ]:

if E2E_DIR.exists():
    case_dirs = [p for p in E2E_DIR.iterdir() if p.is_dir()]
    rows = []
    for case_dir in case_dirs:
        before_dir = case_dir / "before"
        after_dir = case_dir / "after"
        if not before_dir.exists() or not after_dir.exists():
            continue

        # обязательные фото
        for slot in ["front", "rear", "side"]:
            before_img = next(iter(sorted(before_dir.glob(f"{slot}*"))), None)
            after_img = next(iter(sorted(after_dir.glob(f"{slot}*"))), None)
            if before_img and after_img:
                before_summary = summarize_image(before_img, expected_slot=slot)
                after_summary = summarize_image(after_img, expected_slot=slot)
                new_dets = compare_detections(before_summary["damage"], after_summary["damage"])
                rows.append({
                    "case_id": case_dir.name,
                    "slot": slot,
                    "before_slot_ok": before_summary.get("slot_ok"),
                    "after_slot_ok": after_summary.get("slot_ok"),
                    "before_quality_accept": before_summary["quality"]["accepted"],
                    "after_quality_accept": after_summary["quality"]["accepted"],
                    "new_damage_count": len(new_dets),
                })
    df = pd.DataFrame(rows)
    display(df.head())
    out_path = DAMAGE_SEG_ROOT / "reports" / "e2e_case_results.csv"
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)
else:
    print("Create ml/data/e2e_cases first, then rerun this notebook.")